# Qualitative Error Analysis — Egyptian Arabic (EG) Dev Set

We manually inspect the 20 lowest sentence-level spBLEU predictions on EG dev, and
quantify how often the model under- or over-generates the English
code-switching that appears in the dialectal references.

**Inputs required** (JSONL, one object per line):
- A predictions file (the shape `models/infer.py --data-source dev` writes):
  `{"conv_id": str, "country": str, "turns": [{"turn_order": int, "prediction": str}, ...]}`
- A references file with the source sentence alongside the reference, per turn:
  `{"conv_id": str, "country": str, "turns": [{"turn_order": int, "sentence": str, "reference": str}, ...]}`

Set `PREDICTIONS_PATH` / `REFERENCES_PATH` below to your own files.

## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# Show full sentence/prediction/reference text instead of truncating columns.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Load predictions and references

In [ ]:
PREDICTIONS_PATH = "development_predictions.jsonl"  # from models/infer.py --data-source dev
REFERENCES_PATH = "development_references.jsonl"      # sentence + reference per turn (see format above)


def load_jsonl_to_df(file_path: str, is_prediction: bool) -> pd.DataFrame:
    """
    Flatten a conversations-JSONL file into one row per turn.

    is_prediction=True  expects each turn to have "prediction".
    is_prediction=False expects each turn to have "sentence" and "reference".
    """
    all_turns = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            conv_id = data["conv_id"]
            country = data["country"]

            for turn in data["turns"]:
                turn_data = {
                    "conv_id": conv_id,
                    "country": country,
                    "turn": turn["turn_order"],
                }
                if is_prediction:
                    turn_data["prediction"] = turn["prediction"]
                else:
                    turn_data["sentence"] = turn["sentence"]
                    turn_data["reference"] = turn["reference"]

                all_turns.append(turn_data)

    return pd.DataFrame(all_turns)


pred_df = load_jsonl_to_df(PREDICTIONS_PATH, is_prediction=True)
ref_df = load_jsonl_to_df(REFERENCES_PATH, is_prediction=False)

analysis_df = pd.merge(pred_df, ref_df, on=["conv_id", "country", "turn"], how="inner")
analysis_df = analysis_df[["conv_id", "country", "turn", "sentence", "prediction", "reference"]]

analysis_df.head()

,conv_id,country,turn,sentence,prediction,reference
0,B2-1-0-260,EG,1,"Good morning. The best cucumbers and peppers you'll find today, fresh from the farm.",صباح الخير. احسن خيار وفلفل هتلاقيهم النهارده، طازة من المزرعة.,صباح الخير. خيار وفلفل من وش القفص، من الارض على هنا على طول
1,B2-1-0-260,EG,2,Good morning to you. What's the final price for the whole lot?,صباح النور. ايه اخر سعر للدفعة كلها؟,صباح الفل عليك. السعر الاخير للشكارة كلها كام؟
2,B2-1-0-260,EG,3,"For you, a special price. Let's say eight pounds a kilo, and may God bless the sale.",عشانك، سعر مخصوص. خلينا نقول تمنية جنيه للكيلو، وربنا يبارك في البيع.,ليك انت سعر مخصوص. خلينا نقول تمانية جنيه للكيلو، وربنا يبارك في البيعة
3,B2-1-0-260,EG,4,The market is flooded today. I'll take them all for seven. It's a fair price for both of us.,السوق مليان خيرات. هاخدهم كلهم بسبعة. ده سعر عادل لينا احنا التنين.,السوق مليان النهاردة، انا هاخد الكليو بسبعة. كدة احنا الاتنين مرضيين.
4,B9-1-0-189,EG,1,We need to discuss aligning our water policies with your ministry's strategy for water-intensive crops like rice and sugarcane.,إحنا محتاجين نناقش تطابق سياسات المية بتاعتنا مع استراتيجية وزارة الزراعة في المحاصيل اللي بتستهلك مية كتير زي الرز والسكر.,محتاجين ننسق سياسات المياه بتاعتنا مع استراتيجية وزارتكم بخصوص المحاصيل الشرهة للمية زي الرز وقصب السكر.


## Code-switching under-generation

Flag dev turns where the reference contains Latin-script tokens (English
code-switching, e.g. technical/borrowed terms) but the model's prediction
renders everything in Arabic script — the pattern described in the paper as
a general bias toward suppressing code-switching.

In [4]:
under_generation_df = analysis_df[
    analysis_df["reference"].str.contains(r"[a-zA-Z]", na=False)
    & ~analysis_df["prediction"].str.contains(r"[a-zA-Z]", na=False)
].copy()

under_generation_df["country"].value_counts()

country
MA    319
LB    148
TN    123
EG     70
MR     53
SA     28
OM     15
JO     13
PS     13
SY      8
YE      2
Name: count, dtype: int64

## Code-switching over-generation (reverse case)

The opposite pattern — Latin script appears in the prediction but not in the
reference — for comparison. Per the paper, this is comparatively rare except
for Tunisian.

In [5]:
over_generation_df = analysis_df[
    analysis_df["prediction"].str.contains(r"[a-zA-Z]", na=False)
    & ~analysis_df["reference"].str.contains(r"[a-zA-Z]", na=False)
].copy()

over_generation_df["country"].value_counts()

country
TN    212
MA     52
LB     43
EG     12
PS      8
JO      7
OM      5
SA      5
YE      4
SY      2
MR      1
Name: count, dtype: int64

## Sentence-level spBLEU scoring

In [6]:
def add_spbleu_scores_to_dataframe(df: pd.DataFrame, n: int | None = None) -> pd.DataFrame:
    """
    Compute sentence-level spBLEU (flores200 tokenizer) for every row and
    return the DataFrame sorted from lowest to highest score.

    Parameters
    ----------
    df : DataFrame with "prediction" and "reference" columns.
    n  : if set, return only the lowest-n scoring rows; otherwise return all.
    """
    try:
        from sacrebleu.metrics import BLEU
    except ImportError as exc:
        raise ImportError("Install scoring dependencies with: pip install sacrebleu sentencepiece") from exc

    scored_df = df.copy()

    try:
        bleu = BLEU(tokenize="flores200", effective_order=True)
    except Exception as exc:
        raise RuntimeError(
            "Could not initialize SacreBLEU's flores200 tokenizer. "
            "Install/upgrade sacrebleu and sentencepiece."
        ) from exc

    scores = [
        round(bleu.sentence_score(str(pred), [str(ref)]).score, 6)
        for pred, ref in tqdm(
            zip(scored_df["prediction"], scored_df["reference"]),
            total=len(scored_df), desc="Scoring sentences",
        )
    ]
    scored_df["spbleu"] = scores

    sorted_df = scored_df.sort_values(by="spbleu", ascending=True)
    if n is not None:
        sorted_df = sorted_df.head(n)

    return sorted_df.reset_index(names="original_dataframe_index")

## The 20 lowest-scoring EG examples

Manually inspected to identify recurring failure patterns (over-literal
translation of routine social exchanges; under-generation of code-switched
terms — see the sections above for the aggregate counts).

In [7]:
EG_df = analysis_df[analysis_df["country"] == "EG"]
lowest_20_eg = add_spbleu_scores_to_dataframe(EG_df, 20)
lowest_20_eg

Scoring sentences: 100%|██████████| 1113/1113 [00:00<00:00, 6962.34it/s]


,original_dataframe_index,conv_id,country,turn,sentence,prediction,reference,spbleu
0,13,B6-1-0-540,EG,3,"Alright, the corner spot it is. God willing, it will be a profitable day for everyone.",تمام، المكان بتاع الزاوية. إن شاء الله يبقى يوم مربح للكل.,خلاص على الركن. ربنا يرزقنا جميعا.,2.159701
1,839,B7-1-7-201,EG,3,And the developer will issue this to me or the seller?,المطور هيبعتهولي أنا أو للمشتري؟,والسمسار هيطلع الخطاب ده ليا ولا للبايع؟,2.452471
2,1079,B9-1-9-84,EG,2,"You're right, we do. What's your idea?",معاك حق، احنا بنعمل كده. ايه الفكرة بتاعتك؟,عندك حق فعلا، إيه فكرتك؟,2.826521
3,662,B6-1-5-300,EG,2,"Not at all, please, go ahead. I hope everything is okay.",لا خالص، تفضلي. عايزة أعرف كل حاجة تمام؟,اتفضل—خير؟,2.839839
4,959,B2-1-8-506,EG,1,"Alright, what's all the shouting about? Let's be calm.",طيب، إيه الصوت العالي ده؟ خلينا هاديين.,بالراحة الزعيق ده كله على ايه؟ استهدوا بالله,2.908318
5,452,B9-1-3-487,EG,2,"It's about quality, not quantity. Aim for one or two papers per day, solved under exam conditions. The most important part is to review your mistakes afterwards.",الموضوع مش في العدد، المهم الجودة. حاولي ورقتين في اليوم، وتحليهم في ظروف الامتحان. أهم حاجة بعد ما تخلصي إنك تراجعي غلطاتك.,الموضوع جودة مش كمية، امتحان أو اتنين في اليوم تحت ظروف حقيقية، والأهم تراجع أخطائك بعدين.,3.087629
6,711,B6-1-6-406,EG,3,"It's a ten, a definite ten! I can't take it anymore.",ده عشرة، عشرة اكيدة! مش قادر اتحمله.,١٠—مش قادر أستحمل.,3.125191
7,499,B9-1-3-89,EG,3,I just feel so much pressure from my family and from myself to finish.,أنا بس عندي ضغط كبير من العيلة ومن نفسي إني أخلص.,انا حاسة اني مضغوطة اوي من نفسي ومن أهلي.,3.249047
8,710,B6-1-6-406,EG,2,"On a scale of one to ten, how bad is the pain?",على مقياس من واحد لعشرة، قد ايه الالم؟,من ١ ل١٠ الألم كام؟,3.377156
9,37,B7-1-0-342,EG,1,"So, who are you voting for, sister?",طيب، بتصوتي لحد يا أختي؟,هتدي صوتك لمين يا حبيبتي ؟,3.386499


In [8]:
for _, ex in lowest_20_eg.iterrows():
    print(f"Sentence: {ex['sentence']}\nPrediction: {ex['prediction']}\nReference: {ex['reference']}\nspBLEU Score: {ex['spbleu']}\n")

Sentence: Alright, the corner spot it is. God willing, it will be a profitable day for everyone.
Prediction: تمام، المكان بتاع الزاوية. إن شاء الله يبقى يوم مربح للكل.
Reference: خلاص على الركن. ربنا يرزقنا جميعا.
spBLEU Score: 2.159701

Sentence: And the developer will issue this to me or the seller?
Prediction: المطور هيبعتهولي أنا أو للمشتري؟
Reference: والسمسار هيطلع الخطاب ده ليا ولا للبايع؟
spBLEU Score: 2.452471

Sentence: You're right, we do. What's your idea?
Prediction: معاك حق، احنا بنعمل كده. ايه الفكرة بتاعتك؟
Reference: عندك حق فعلا، إيه فكرتك؟
spBLEU Score: 2.826521

Sentence: Not at all, please, go ahead. I hope everything is okay.
Prediction: لا خالص، تفضلي. عايزة أعرف كل حاجة تمام؟
Reference: اتفضل—خير؟
spBLEU Score: 2.839839

Sentence: Alright, what's all the shouting about? Let's be calm.
Prediction: طيب، إيه الصوت العالي ده؟ خلينا هاديين.
Reference: بالراحة الزعيق ده كله على ايه؟ استهدوا بالله
spBLEU Score: 2.908318

Sentence: It's about quality, not quantity. Aim 